In [0]:
%run ./00_config

In [0]:
dbutils.widgets.text("fecha", "todas")          # "todas" = backfill; el job pasa YYYY-MM-DD
fecha = dbutils.widgets.get("fecha")
print("fecha:", fecha)

SALTO_KN = 60.0   # velocidad implícita entre dos posiciones por encima de esto = salto anómalo

tbl_silver = T("silver", "ais_posiciones")
tbl_traf   = T("gold", "trafico_celda_hora")
tbl_tray   = T("gold", "trayectorias_diarias")

In [0]:
from pyspark.sql import functions as F, Window

# "todas" = backfill; si no, una fecha o una lista separada por comas (llegan varios zips a la vez)
fechas = [f.strip() for f in fecha.split(",")]
pred = "true" if fecha == "todas" else "fecha IN (" + ", ".join(f"DATE'{f}'" for f in fechas) + ")"
print("filtro:", pred)
s = spark.table(tbl_silver).filter(pred)

def escribir(df, tbl, cluster_cols):
    """Idempotente: sobrescribe solo las fechas procesadas."""
    if not spark.catalog.tableExists(tbl):
        df.write.saveAsTable(tbl)
        spark.sql(f"ALTER TABLE {tbl} CLUSTER BY ({', '.join(cluster_cols)})")
    else:
        df.write.mode("overwrite").option("replaceWhere", pred).saveAsTable(tbl)
    print(tbl, "→", spark.table(tbl).filter(pred).count(), "filas para", fecha)

def show(df, n=10):
    print(df.limit(n).toPandas().to_string(index=False))

In [0]:
for t in [tbl_silver, tbl_traf, tbl_tray, T("silver", "ref_puerto_celdas"), T("gold", "presencia_puertos")]:
    print(t, spark.catalog.tableExists(t))

In [0]:
centro = F.expr("h3_centerasgeojson(h3_r7)")
traf = (s.withColumn("hora", F.date_trunc("hour", "ts"))
         .groupBy("fecha", "hora", "h3_r7")
         .agg(F.count("*").alias("posiciones"),
              F.countDistinct("mmsi").alias("buques"),
              F.round(F.avg("sog"), 2).alias("sog_promedio"))
         .withColumn("lat", F.get_json_object(centro, "$.coordinates[1]").cast("double"))
         .withColumn("lon", F.get_json_object(centro, "$.coordinates[0]").cast("double")))
escribir(traf, tbl_traf, ["fecha", "h3_r7"])

In [0]:
w = Window.partitionBy("mmsi", "fecha").orderBy("ts")
R = 6371.0
seg = (s.filter("es_buque")
        .withColumn("lat0", F.lag("lat").over(w))
        .withColumn("lon0", F.lag("lon").over(w))
        .withColumn("dt_h", (F.col("ts").cast("long") - F.lag("ts").over(w).cast("long")) / 3600)
        .withColumn("d_km", 2 * R * F.asin(F.sqrt(
            F.pow(F.sin(F.radians(F.col("lat") - F.col("lat0")) / 2), 2) +
            F.cos(F.radians("lat0")) * F.cos(F.radians("lat")) *
            F.pow(F.sin(F.radians(F.col("lon") - F.col("lon0")) / 2), 2))))
        .withColumn("v_impl_kn", F.when(F.col("dt_h") > 0, F.col("d_km") / 1.852 / F.col("dt_h")))
        .withColumn("es_salto", F.coalesce((F.col("v_impl_kn") > SALTO_KN) & (F.col("d_km") > 1.0), F.lit(False))))
tray = (seg.groupBy("mmsi", "fecha")
           .agg(F.count("*").alias("posiciones"),
                F.min("ts").alias("ts_inicio"), F.max("ts").alias("ts_fin"),
                F.min_by("lat", "ts").alias("lat_inicio"), F.min_by("lon", "ts").alias("lon_inicio"),
                F.max_by("lat", "ts").alias("lat_fin"), F.max_by("lon", "ts").alias("lon_fin"),
                F.round(F.sum(F.when(~F.col("es_salto"), F.col("d_km"))), 2).alias("distancia_km"),
                F.round(F.avg("sog"), 2).alias("sog_promedio"),
                F.max("sog").alias("sog_max"),
                F.sum(F.col("es_salto").cast("int")).alias("saltos_anomalos"),
                F.round(F.max("v_impl_kn"), 1).alias("v_impl_max_kn"),
                F.max("vessel_name").alias("vessel_name"),
                F.max("vessel_type").alias("vessel_type"),
                F.max("length").alias("length"))
           .withColumn("horas_activo", F.round((F.col("ts_fin").cast("long") - F.col("ts_inicio").cast("long")) / 3600, 2)))
escribir(tray, tbl_tray, ["fecha", "mmsi"])

In [0]:
print("-- Top 10 celdas-hora por buques")
show(spark.table(tbl_traf).filter(pred).orderBy(F.desc("buques")))

print("\n-- Trayectorias: resumen")
show(spark.table(tbl_tray).filter(pred).agg(
    F.count("*").alias("buque_dias"),
    F.round(F.sum("distancia_km")).alias("km_totales"),
    F.sum((F.col("saltos_anomalos") > 0).cast("int")).alias("buque_dias_con_saltos"),
    F.sum("saltos_anomalos").alias("saltos_totales")))

print("\n-- Top 10 buques con más saltos anómalos")
show(spark.table(tbl_tray).filter(pred).orderBy(F.desc("saltos_anomalos"))
     .select("fecha", "mmsi", "vessel_name", "posiciones", "saltos_anomalos", "v_impl_max_kn", "distancia_km"))

In [0]:
import os, shutil, json, urllib.request, urllib.parse

R_KM, K = 5.0, 3    # radio de puerto; k-ring 3 en res 7 cubre ~6 km
tbl_pc   = T("silver", "ref_puerto_celdas")
tbl_pres = T("gold", "presencia_puertos")

# 1. Archivo WPI en el volume (se copia de una fuente local o se descarga de NGA)
wpi_path = f"{V('ref')}/WorldPortIndex.csv"
candidatos = ["/Volumes/workspace/default/proyecto/WorldPortIndex.csv",   # copia de la Entrega 1 (Free Edition)
              os.path.abspath("../../WorldPortIndex.csv")]               # CSV en la raíz del repo (Git folder)
if not os.path.exists(wpi_path):
    local = next((c for c in candidatos if os.path.exists(c)), None)
    if local:
        shutil.copy(local, wpi_path); print("WPI copiado de", local)
    else:
        url = ("https://services2.arcgis.com/jUpNdisbWqRpMo35/ArcGIS/rest/services/WPI_Ports2017/FeatureServer/0/query?"
               + urllib.parse.urlencode({"where": "1=1", "outFields": "*", "outSR": 4326, "f": "json"}))
        feats = json.load(urllib.request.urlopen(url, timeout=60))["features"]
        import pandas as pd
        pd.DataFrame([{**f["attributes"], "LONGITUDE": f["geometry"]["x"], "LATITUDE": f["geometry"]["y"]}
                      for f in feats]).to_csv(wpi_path, index=False)
        print("WPI descargado")
print("WPI:", wpi_path)

# 2. Puertos con coordenadas
puertos = (spark.read.csv(wpi_path, header=True, inferSchema=True)
           .select(F.col("PORT_NAME").alias("puerto"), F.col("COUNTRY").alias("pais"),
                   F.col("LATITUDE").cast("double").alias("lat_puerto"),
                   F.col("LONGITUDE").cast("double").alias("lon_puerto"))
           .dropna(subset=["lat_puerto", "lon_puerto"]))
print("puertos WPI:", puertos.count())

# 3. Celdas H3 a <= R_KM de cada puerto (si una celda queda cerca de dos puertos, gana el más cercano)
def haversine(lat1, lon1, lat2, lon2):
    return 2 * 6371 * F.asin(F.sqrt(F.pow(F.sin(F.radians(lat2 - lat1) / 2), 2) +
           F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.pow(F.sin(F.radians(lon2 - lon1) / 2), 2)))

centro = F.expr("h3_centerasgeojson(h3_r7)")
pc = (puertos
      .withColumn("h3_r7", F.explode(F.expr(f"h3_kring(h3_longlatash3string(lon_puerto, lat_puerto, 7), {K})")))
      .withColumn("lat_c", F.get_json_object(centro, "$.coordinates[1]").cast("double"))
      .withColumn("lon_c", F.get_json_object(centro, "$.coordinates[0]").cast("double"))
      .withColumn("dist_km", haversine(F.col("lat_puerto"), F.col("lon_puerto"), F.col("lat_c"), F.col("lon_c")))
      .filter(F.col("dist_km") <= R_KM)
      .withColumn("rn", F.row_number().over(Window.partitionBy("h3_r7").orderBy("dist_km")))
      .filter("rn = 1")
      .select("h3_r7", "puerto", "pais", "lat_puerto", "lon_puerto", F.round("dist_km", 2).alias("dist_km")))
pc.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl_pc)
print("celdas de puerto:", spark.table(tbl_pc).count())

In [0]:
zonas = spark.table(tbl_pc).select("h3_r7", "puerto", "pais", "lat_puerto", "lon_puerto")
w2 = Window.partitionBy("mmsi", "fecha", "puerto").orderBy("ts")

pres = (s.filter("es_buque AND sog IS NOT NULL AND sog <= 1.0")
         .join(F.broadcast(zonas), "h3_r7")
         .withColumn("dt_h", (F.col("ts").cast("long") - F.lag("ts").over(w2).cast("long")) / 3600)
         .groupBy("fecha", "mmsi", "puerto", "pais")
         .agg(F.count("*").alias("posiciones"),
              F.min("ts").alias("primera_posicion"),
              F.max("ts").alias("ultima_posicion"),
              F.round(F.sum(F.when(F.col("dt_h") <= 1, F.col("dt_h")).otherwise(0)), 2).alias("horas_en_puerto"),
              F.max("vessel_name").alias("vessel_name"),
              F.max("vessel_type").alias("vessel_type"),
              F.first("lat_puerto").alias("lat_puerto"),
              F.first("lon_puerto").alias("lon_puerto")))
escribir(pres, tbl_pres, ["fecha", "puerto"])

In [0]:
p = spark.table(tbl_pres).filter(pred)
print("-- Top 15 puertos por buques distintos (todo el período)")
show(p.groupBy("puerto", "pais").agg(F.countDistinct("mmsi").alias("buques"),
                                     F.round(F.sum("horas_en_puerto")).alias("horas_buque"))
      .orderBy(F.desc("buques")), 15)

print("\n-- Buques en puerto por día")
show(p.groupBy("fecha").agg(F.countDistinct("mmsi").alias("buques_en_puerto"),
                            F.countDistinct("puerto").alias("puertos_activos")).orderBy("fecha"))